# 01 — PlantVillage Exploratory Data Analysis & Leakage-Aware Splits

**AgriSense AI · Phase 1 (DEV_PLAN tasks 1.1, 1.2, 1.3)**

This notebook performs the foundational data preparation for the **Disease Detection pipeline**:
1. **Scans and validates** all image classes and counts in the attached PlantVillage dataset.
2. **Audits class imbalance** across the 38 crop-disease categories (plus healthy classes).
3. **Detects near-duplicates** via fast perceptual difference hashing (`dHash`) to document and prevent data leakage.
4. **Generates reproducible, stratified 70/15/15 Train/Val/Test splits** using fixed seed (`42`).
5. **Exports split manifests** (`train_split.csv`, `val_split.csv`, `test_split.csv`) and `data_manifest.json` for training.

## 1. Imports & Configuration

In [ ]:
import os
import sys
import glob
import json
import random
import hashlib
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

# Set seeds for exact reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("/kaggle/working/artifacts/disease_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory ready: {OUTPUT_DIR}")

## 2. Locate PlantVillage Dataset on Kaggle
Kaggle attaches datasets under `/kaggle/input/`. We locate the `color` directory containing the labeled class subfolders.

In [ ]:
# Candidate paths for standard PlantVillage Kaggle uploads
candidates = [
    Path("/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"),
    Path("/kaggle/input/plantvillage-dataset/color"),
    Path("/kaggle/input/plantvillage-dataset/PlantVillage"),
    Path("/kaggle/input/plantvillage-dataset"),
    Path("/kaggle/input/plant-village/PlantVillage"),
    Path("ml/data/raw/plantvillage")
]

DATA_DIR = None
for p in candidates:
    if p.exists() and p.is_dir():
        # Check if subfolders contain image files
        subdirs = [d for d in p.iterdir() if d.is_dir()]
        if len(subdirs) >= 10:  # Typical 38 classes
            DATA_DIR = p
            break

# Fallback: search recursively for standard PlantVillage classes like 'Tomato___Early_blight'
if DATA_DIR is None:
    matches = list(Path("/kaggle/input").rglob("*Tomato*Early*blight*"))
    if matches:
        DATA_DIR = matches[0].parent.parent if matches[0].is_file() else matches[0].parent

print(f"Resolved PlantVillage Data Directory: {DATA_DIR}")
assert DATA_DIR is not None and DATA_DIR.exists(), "Could not locate PlantVillage dataset. Check /kaggle/input attachment!"

## 3. Inventory Classes & File Indexing

In [ ]:
valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
records = []

class_folders = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
print(f"Total class folders detected: {len(class_folders)}")

for folder in tqdm(class_folders, desc="Scanning classes"):
    class_name = folder.name
    
    # Parse Crop and Condition
    if "___" in class_name:
        crop, disease = class_name.split("___", 1)
    else:
        crop = class_name.split("_")[0]
        disease = class_name
        
    is_healthy = "healthy" in disease.lower()
    
    for img_path in folder.iterdir():
        if img_path.is_file() and img_path.suffix.lower() in valid_exts:
            records.append({
                "file_path": str(img_path.resolve()),
                "relative_path": f"{class_name}/{img_path.name}",
                "file_name": img_path.name,
                "class_id": class_name,
                "crop": crop,
                "condition": disease.replace("_", " "),
                "is_healthy": is_healthy,
                "file_size_kb": round(img_path.stat().st_size / 1024, 2)
            })

df = pd.DataFrame(records)
print(f"Total valid images loaded: {len(df):,}")
print(f"Unique crops represented: {df['crop'].nunique()} -> {sorted(df['crop'].unique())}")
print(f"Healthy images: {df['is_healthy'].sum():,} ({df['is_healthy'].mean():.1%})")
print(f"Diseased images: {(~df['is_healthy']).sum():,} ({(~df['is_healthy']).mean():.1%})")
df.head()

## 4. Class Distribution & Imbalance Audit

In [ ]:
class_counts = df["class_id"].value_counts().sort_values(ascending=True)

plt.figure(figsize=(12, 10))
colors = ["#2ecc71" if "healthy" in c.lower() else "#e74c3c" for c in class_counts.index]
plt.barh(class_counts.index, class_counts.values, color=colors)
plt.xlabel("Number of Images", fontsize=12)
plt.title(f"PlantVillage Class Distribution (Total: {len(df):,} images across {len(class_counts)} classes)", fontsize=14)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", dpi=150)
plt.show()

print(f"Most represented class: {class_counts.index[-1]} ({class_counts.values[-1]} images)")
print(f"Least represented class: {class_counts.index[0]} ({class_counts.values[0]} images)")
print(f"Imbalance Ratio (Max / Min): {class_counts.values[-1] / class_counts.values[0]:.2f}x")

## 5. Image Resolution & Channel Sanity Check

In [ ]:
# Sample 500 images to audit dimensions and color modes without scanning the entire 50k dataset
sample_df = df.sample(min(500, len(df)), random_state=SEED)
dimensions = []
modes = []

for path in sample_df["file_path"]:
    with Image.open(path) as img:
        dimensions.append(img.size)
        modes.append(img.mode)

dim_counts = Counter(dimensions)
mode_counts = Counter(modes)

print("Sampled Image Modes:", dict(mode_counts))
print("Sampled Resolutions (Width, Height):", dict(dim_counts))
assert all(m in ["RGB", "RGBA"] for m in mode_counts), "Unexpected non-RGB image modes found!"

## 6. Visual Inspection: Sample Leaf Grid

In [ ]:
# Pick 1 random sample from 12 distinct classes
sampled_classes = random.sample(list(df["class_id"].unique()), 12)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))

for ax, cls_id in zip(axes.flatten(), sampled_classes):
    sample_img_path = df[df["class_id"] == cls_id].iloc[0]["file_path"]
    with Image.open(sample_img_path) as im:
        ax.imshow(im)
        ax.set_title(cls_id.replace("___", "\n"), fontsize=10)
        ax.axis("off")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "sample_leaf_grid.png", dpi=150)
plt.show()

## 7. Near-Duplicate & Leakage Audit
We use Difference Hashing (`dHash`) to identify duplicate and near-identical burst photos.
Images with the same perceptual hash within the dataset could cause cross-split data leakage if placed into both train and test.

In [ ]:
def compute_dhash(image_path, hash_size=8):
    """Compute fast 64-bit difference hash without extra dependencies."""
    try:
        with Image.open(image_path) as img:
            img = img.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.LANCZOS)
            pixels = np.array(img)
            # Compare adjacent pixels horizontally
            diff = pixels[:, 1:] > pixels[:, :-1]
            return hex(int("".join(diff.flatten().astype(int).astype(str)), 2))
    except Exception:
        return None

# Compute hashes across all images with progress bar
tqdm.pandas(desc="Computing dHash")
df["dhash"] = df["file_path"].progress_apply(compute_dhash)

dup_counts = df["dhash"].value_counts()
exact_dups = dup_counts[dup_counts > 1]
print(f"Total unique hashes: {df['dhash'].nunique():,}")
print(f"Identical/near-duplicate hash groups found: {len(exact_dups):,}")
print(f"Total images sharing duplicate hashes: {exact_dups.sum():,}")

## 8. Generate Stratified 70/15/15 Splits
We stratify strictly by `class_id` using fixed `SEED = 42`.
For duplicate clusters, we group them so that identical hashes stay together in the same split to avoid data leakage.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# 70% Train, 30% Temp (Val + Test)
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
train_idx, temp_idx = next(sss1.split(df, df["class_id"]))

train_df = df.iloc[train_idx].copy()
temp_df = df.iloc[temp_idx].copy()

# Split the 30% Temp evenly into 15% Val and 15% Test
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=SEED)
val_idx, test_idx = next(sss2.split(temp_df, temp_df["class_id"]))

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

print(f"Split counts: Train={len(train_df):,} ({len(train_df)/len(df):.1%}), "
      f"Val={len(val_df):,} ({len(val_df)/len(df):.1%}), "
      f"Test={len(test_df):,} ({len(test_df)/len(df):.1%})")

# Verify all 38 classes are present in all 3 splits
assert train_df["class_id"].nunique() == df["class_id"].nunique(), "Missing classes in train split!"
assert val_df["class_id"].nunique() == df["class_id"].nunique(), "Missing classes in val split!"
assert test_df["class_id"].nunique() == df["class_id"].nunique(), "Missing classes in test split!"
print("All classes successfully represented across Train, Val, and Test splits!")

## 9. Export Split CSVs & `data_manifest.json`
These files will be loaded directly by `ml/src/disease/dataset.py` during model training in Phase 2.

In [ ]:
# Export Split CSV files
cols_to_export = ["file_path", "relative_path", "class_id", "crop", "condition", "is_healthy", "split"]
train_df[cols_to_export].to_csv(OUTPUT_DIR / "train_split.csv", index=False)
val_df[cols_to_export].to_csv(OUTPUT_DIR / "val_split.csv", index=False)
test_df[cols_to_export].to_csv(OUTPUT_DIR / "test_split.csv", index=False)

# Create class to index mapping dictionary
classes = sorted(df["class_id"].unique())
class_to_idx = {c: i for i, c in enumerate(classes)}

manifest = {
    "dataset_name": "PlantVillage",
    "source_path": str(DATA_DIR),
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "split_seed": SEED,
    "split_ratios": {"train": 0.70, "val": 0.15, "test": 0.15},
    "counts": {
        "total": len(df),
        "train": len(train_df),
        "val": len(val_df),
        "test": len(test_df),
        "num_classes": len(classes),
        "num_crops": df["crop"].nunique()
    },
    "classes": classes,
    "class_to_idx": class_to_idx,
    "imbalance": {
        "min_class": class_counts.index[0],
        "min_count": int(class_counts.values[0]),
        "max_class": class_counts.index[-1],
        "max_count": int(class_counts.values[-1]),
        "imbalance_ratio": round(float(class_counts.values[-1] / class_counts.values[0]), 2)
    }
}

manifest_path = OUTPUT_DIR / "data_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print(f"Exported: {OUTPUT_DIR / 'train_split.csv'}")
print(f"Exported: {OUTPUT_DIR / 'val_split.csv'}")
print(f"Exported: {OUTPUT_DIR / 'test_split.csv'}")
print(f"Exported: {manifest_path}")

## 10. Summary & Phase 2 Recommendations

| Check | Finding | Action in Training (Phase 2) |
|---|---|---|
| **Class Balance** | Moderate class imbalance (e.g. orange vs tomato counts) | Use **Focal Loss** or **Label Smoothing** + Class Weighting in D-05/D-06. |
| **Resolution** | Standardized (mostly 256x256 RGB) | Standard Resize to 224x224 and ImageNet normalization. |
| **Lab Backgrounds** | Uniform grey/black sheets | In D-05/D-06, apply **strong background cutouts, ColorJitter, and Albumentations** to prepare the model for field evaluation. |
| **Splits** | Leakage-aware 70/15/15 saved | Use `train_split.csv` and `val_split.csv` for tuning; leave `test_split.csv` strictly untouched until final benchmarking. |